<a href="https://colab.research.google.com/github/Harsh-Shah-04/RSA_Final_Practical/blob/main/D24AIML085_RSA_Final_Exam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df = pd.read_csv('/content/german_credit_data.csv')
print("Columns:", df.columns.tolist())
display(df.head())

Columns: ['Unnamed: 0', 'Age', 'Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account', 'Credit amount', 'Duration', 'Purpose']


,Unnamed: 0,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose
0,0,67,male,2,own,NaN,little,1169,6,radio/TV
1,1,22,female,2,own,little,moderate,5951,48,radio/TV
2,2,49,male,1,own,little,NaN,2096,12,education
3,3,45,male,2,free,little,little,7882,42,furniture/equipment
4,4,53,male,2,free,little,little,4870,24,car


In [ ]:
import pandas as pd
import numpy as np

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data'
columns = [
    'checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount',
    'savings_status', 'employment', 'installment_commitment', 'personal_status',
    'other_parties', 'residence_since', 'property_magnitude', 'age',
    'other_payment_plans', 'housing', 'existing_credits', 'job', 'num_dependents',
    'own_telephone', 'foreign_worker', 'class'
]

df = pd.read_csv(url, sep=' ', header=None, names=columns)
df['class'] = df['class'].map({1: 0, 2: 1})

print("Dataset Shape:", df.shape)
print("Class distribution:\n", df['class'].value_counts())
display(df.head())

Dataset Shape: (1000, 21)
Class distribution:
 class
0    700
1    300
Name: count, dtype: int64


,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,0
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,1
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,0
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,0
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,1


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

encoder = OrdinalEncoder()
df_encoded = df.copy()
df_encoded[categorical_cols] = encoder.fit_transform(df[categorical_cols])

X = df_encoded.drop(columns=['class'])
y = df_encoded['class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

rf_model = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print(f"Model: RandomForest (200 trees, depth 8)")
print(f"Test accuracy: {accuracy:.2f} ROC-AUC: {roc_auc:.2f}")

Model: RandomForest (200 trees, depth 8)
Test accuracy: 0.77 ROC-AUC: 0.81


In [ ]:
import shap
import lime
import lime.lime_tabular
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

explainer_shap = shap.TreeExplainer(rf_model)
shap_values = explainer_shap(X_test)
shap_vals_class1 = shap_values.values[:, :, 1] if len(shap_values.values.shape) == 3 else shap_values.values

mean_shap = np.abs(shap_vals_class1).mean(axis=0)
importance_df = pd.DataFrame({
    'FEATURE': X_test.columns,
    'MEAN|SHAP|': mean_shap
}).sort_values(by='MEAN|SHAP|', ascending=False)

print("GLOBAL FEATURE IMPORTANCE (mean |SHAP|, class = bad)")
print(importance_df.head(5).to_string(index=False))

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_vals_class1, X_test, show=False)
plt.title("SHAP Summary Plot (Class = Bad)", fontsize=14)
plt.tight_layout()
plt.savefig('shap_summary.png')
plt.close()
print("\nPlot saved : shap_summary.png")

GLOBAL FEATURE IMPORTANCE (mean |SHAP|, class = bad)
        FEATURE  MEAN|SHAP|
checking_status    0.086658
       duration    0.041341
 savings_status    0.039125
  credit_amount    0.025849
 credit_history    0.025785

Plot saved : shap_summary.png


In [ ]:
test_index = 23
if test_index in X_test.index:
    applicant_idx = test_index
else:
    applicant_idx = X_test.index[23]

applicant = X_test.loc[[applicant_idx]]
pred_prob = rf_model.predict_proba(applicant)[0, 1]
pred_class = rf_model.predict(applicant)[0]

print(f"Applicant (test index {applicant_idx}) : P(bad) = {pred_prob:.2f} -> {'REJECTED' if pred_prob >= 0.5 else 'APPROVED'}")

test_loc = X_test.index.get_loc(applicant_idx)
shap_val_app = shap_values[test_loc, :, 1] if len(shap_values.shape) == 3 else shap_values[test_loc]

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
shap.plots.waterfall(shap_val_app, show=False)
plt.title(f"SHAP Waterfall Plot for Applicant {applicant_idx}", fontsize=14)
plt.tight_layout()
plt.savefig('shap_local.png')
plt.close()
print("Local SHAP plot saved: shap_local.png")

feature_names = X_test.columns
shap_contributions = pd.DataFrame({
    'feature': feature_names,
    'value': applicant.values[0],
    'contribution': shap_val_app.values
})
shap_contributions['abs_contribution'] = shap_contributions['contribution'].abs()
shap_contributions = shap_contributions.sort_values(by='abs_contribution', ascending=False)
print("\nTop SHAP contributions for applicant:")
print(shap_contributions.drop(columns=['abs_contribution']).head(5))

Applicant (test index 570) : P(bad) = 0.61 -> REJECTED
Local SHAP plot saved: shap_local.png

Top SHAP contributions for applicant:
            feature  value  contribution
0   checking_status    0.0      0.114333
1          duration   24.0      0.055410
6        employment    1.0      0.048357
5    savings_status    0.0      0.040876
12              age   23.0      0.035518


In [ ]:
explainer_lime = lime.lime_tabular.LimeTabularExplainer(
    training_data=np.array(X_train),
    feature_names=X_train.columns,
    class_names=['good', 'bad'],
    mode='classification',
    random_state=42
)

exp = explainer_lime.explain_instance(
    data_row=X_test.loc[applicant_idx],
    predict_fn=rf_model.predict_proba,
    num_features=5
)

print("LIME Local Explanation (Class = bad):")
for feature, weight in exp.as_list():
    print(f"{feature}: {weight:+.4f}")

/usr/local/lib/python3.13/dist-packages/lime/discretize.py:110: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ret[feature] = int(self.lambdas[feature](ret[feature]))
/usr/local/lib/python3.13/dist-packages/lime/discretize.py:110: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  ret[feature] = int(self.lambdas[feature](ret[feature]))
/usr/local/lib/python3.13/dist-packages/lime/lime_tabular.py:544: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.

LIME Local Explanation (Class = bad):
checking_status <= 0.00: +0.1262
savings_status <= 0.00: +0.0728
credit_history <= 2.00: +0.0726
employment <= 2.00: +0.0495
3.00 < installment_commitment <= 4.00: +0.0393
